# Gemma 4 E4B — QLoRA on free Colab T4

Trains `google/gemma-4-E4B-it` on `train_mix_v2.jsonl` (3,738 chat-format SFT
examples, ~6.3M tokens: 50% programmatic agent trajectories + 50% SWE-Gym SFT).
The base model scores **0/10** on the dev slice — this LoRA is what has to lift it.

**Run top to bottom, no edits needed** (except the optional `RESUME_FROM` in the
training cell). Time: ~2–2.5 h/epoch on a T4 (estimate). Checkpoints are saved
after each epoch, so a disconnect after epoch 1 still leaves a usable adapter.

**Attention:** `attn_implementation="sdpa"` — the only path that works for Gemma 4.
FlashAttention-2 caps head_dim at 256; Gemma 4's 7 global layers use 512, so FA2
raises at load. The vLLM Triton shared-memory wall (96 KB vs T4's 64 KB) does NOT
apply to HF training — SDPA uses cuDNN/xformers/math kernels. Full reasoning in
the project's NOTES.md (attention-path review, 2026-11-17).


In [ ]:
# 0 — Environment: GPU check + installs
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
!pip install -q -U "transformers>=5.0" peft bitsandbytes accelerate datasets jinja2
import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__)
assert torch.cuda.is_available(), "NO GPU — Runtime → Change runtime type → T4 GPU, then re-run."
print("GPU:", torch.cuda.get_device_name(0), "|",
      f"{torch.cuda.get_device_properties(0).total_memory / 2**30:.1f} GiB")
# If the cell below this one fails with an ImportError, do Runtime → Restart session
# and Run all again — pip installs persist across a restart in the same session.


In [ ]:
# 1 — Data: download the 4 GitHub chunks, reassemble, verify sha256
# (single 29 MB upload fails through the uploader's proxy, hence 4 chunks)
import hashlib, shutil, urllib.request
DATA_BASE = "https://raw.githubusercontent.com/coral-fox-12/gemma4-lora-data/main/train_mix_v2.jsonl.part{}"
EXPECTED_SHA256 = "248f127e7fefcaf1ee241359e716c35c9418801a2f2706be675697f71205fa5f"
EXPECTED_LINES = 3738
DATA_PATH = "/content/train_mix_v2.jsonl"

def fetch_data():
    parts = []
    for i in range(4):
        p = f"/content/train_mix_v2.jsonl.part{i}"
        urllib.request.urlretrieve(DATA_BASE.format(i), p)
        parts.append(p)
    with open(DATA_PATH, "wb") as out:
        for p in parts:
            with open(p, "rb") as f:
                out.write(f.read())
    h = hashlib.sha256(open(DATA_PATH, "rb").read()).hexdigest()
    n = sum(1 for _ in open(DATA_PATH))
    assert h == EXPECTED_SHA256, f"sha256 mismatch: {h}"
    assert n == EXPECTED_LINES, f"line count {n} != {EXPECTED_LINES}"
    print(f"data OK: {DATA_PATH} ({n} examples, sha256 verified)")

try:
    fetch_data()
except Exception as e:
    print("download failed:", e)
    print("Fallback: pick train_mix_v2.jsonl in the file picker.")
    from google.colab import files
    up = files.upload()
    assert "train_mix_v2.jsonl" in up, "upload train_mix_v2.jsonl"
    shutil.move("train_mix_v2.jsonl", DATA_PATH)
    print("upload OK")


In [ ]:
# 2 — Tokenizer + model (4-bit NF4) + LoRA
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_ID = "google/gemma-4-E4B-it"  # ungated on HF — no token / licence step needed
tok = AutoTokenizer.from_pretrained(MODEL_ID)
assert tok.chat_template, "no chat template on the tokenizer!"
print("chat template: OK | vocab:", len(tok))

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.bfloat16,
                         bnb_4bit_use_double_quant=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb, dtype=torch.bfloat16,
    attn_implementation="sdpa",  # FA2 cannot run Gemma4 (global head_dim 512 > 256 cap)
    device_map="auto")
model.config.use_cache = False
try:
    model.config.text_config.use_cache = False
except Exception:
    pass
# text-only training: freeze the vision tower + projector outright
for name, p in model.named_parameters():
    if "vision_tower" in name or "multi_modal_projector" in name:
        p.requires_grad_(False)
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora_cfg = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    # regex: only the language-model linears (vision tower excluded by construction)
    target_modules=r".*language_model.*\.(q_proj|k_proj|v_proj|o_proj|gate_proj|up_proj|down_proj)$")
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()
adapted = [n for n, _ in model.named_modules() if "lora_A" in n]
assert len(adapted) == 42 * 7, f"expected 294 LoRA modules, got {len(adapted)}"
assert all("language_model" in n for n in adapted), "LoRA leaked outside language_model!"
print(f"LoRA on {len(adapted)} modules, all inside language_model ✓")
print(f"VRAM after load: {torch.cuda.memory_allocated() / 2**30:.2f} GiB")


In [ ]:
# 3 — Dataset: render with the chat template, mask prompt tokens
# (loss only on the LAST assistant turn of each example)
import json
MAX_LEN = 3072  # real Gemma4-tokenized lengths, all 3,738 rows: median 2056, p95 2502, p99 2659, max 3321; <=3072 keeps 3737/3738
rows = [json.loads(l) for l in open(DATA_PATH)]
print("rows:", len(rows))

# NOTE (transformers 5.x): apply_chat_template(tokenize=True) returns a
# BatchEncoding dict, NOT a list[int]. Use the string path instead.
def encode(msgs):
    assert msgs[-1]["role"] == "assistant"
    full = tok(tok.apply_chat_template(msgs, tokenize=False),
               add_special_tokens=False)["input_ids"]
    prompt = tok(tok.apply_chat_template(msgs[:-1], tokenize=False,
                     add_generation_prompt=True),
                 add_special_tokens=False)["input_ids"]
    assert full[:len(prompt)] == prompt, "prompt is not a token-prefix"
    return {"input_ids": full,
            "labels": [-100] * len(prompt) + full[len(prompt):],
            "attention_mask": [1] * len(full)}

feats, skipped = [], 0
for r in rows:
    try:
        feats.append(encode(r["messages"]))
    except Exception:
        skipped += 1
print(f"encoded {len(feats)}, skipped {skipped}")
lens = sorted(len(f["input_ids"]) for f in feats)
print(f"tokens: median {lens[len(lens)//2]}, p95 {lens[int(len(lens)*0.95)]}, "
      f"max {lens[-1]}")
feats = [f for f in feats if len(f["input_ids"]) <= MAX_LEN]
print(f"after <={MAX_LEN} filter: {len(feats)} examples")
from datasets import Dataset
ds = Dataset.from_list(feats)
print(ds)


In [ ]:
# 4 — Train (set RESUME_FROM to a checkpoint path to continue after a disconnect)
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq

OUT = "/content/gemma4-e4b-lora"
RESUME_FROM = None  # e.g. "/content/gemma4-e4b-lora/checkpoint-234"
args = TrainingArguments(
    output_dir=OUT,
    num_train_epochs=2,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,  # effective batch 16
    learning_rate=0.0002, lr_scheduler_type="cosine", warmup_ratio=0.03,
    bf16=True, gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="paged_adamw_8bit", max_grad_norm=1.0, logging_steps=10,
    save_strategy="epoch", save_total_limit=3, report_to=[], seed=42,
    dataloader_num_workers=2)
trainer = Trainer(model=model, args=args, train_dataset=ds,
                  data_collator=DataCollatorForSeq2Seq(
                      tok, padding=True, label_pad_token_id=-100))
trainer.train(resume_from_checkpoint=RESUME_FROM)
trainer.save_model(OUT + "/final")
tok.save_pretrained(OUT + "/final")
print("saved:", OUT + "/final")
print(f"peak VRAM: {torch.cuda.max_memory_allocated() / 2**30:.2f} GiB")


In [ ]:
# 5 — Package the adapter + download it
import os, shutil
shutil.make_archive("/content/gemma4-e4b-lora", "zip", "/content", "gemma4-e4b-lora")
print(f"zip: {os.path.getsize('/content/gemma4-e4b-lora.zip') / 2**20:.0f} MB")
from google.colab import files
files.download("/content/gemma4-e4b-lora.zip")
print("Download started. Also run the Drive cell below for a backup copy.")


In [ ]:
# 6 — OPTIONAL: save a copy to Google Drive (2 TB on your Pro account)
from google.colab import drive
drive.mount("/content/drive")
!cp -r /content/gemma4-e4b-lora "/content/drive/MyDrive/gemma4-e4b-lora"
print("saved to Drive: MyDrive/gemma4-e4b-lora")


## What happens next

- Send `gemma4-e4b-lora.zip` (or the Drive path) to the agent. It evaluates the
  adapter on Kaggle L4x4 against the 10-task dev slice — target: clearly above the
  0/10 base. Kill criteria: scaffold v1 < 0.20 by ~24/11, or no LoRA gain by
  ~27/11 → stop.
- If Colab disconnects mid-run: the per-epoch checkpoints under
  `/content/gemma4-e4b-lora/checkpoint-*` survive only if you copy them to Drive
  (or download them) before the session dies. To resume, copy the checkpoint back
  to `/content/`, set `RESUME_FROM` in cell 4, and re-run from cell 4.
- If the T4 route fails outright, the fallback is the Kaggle GPU refresh (21/11,
  L4x4) — same data, same config, this notebook's training cell ports 1:1.
